# End-to-End Persian VITS/Piper GPU Fine-Tuning & Evaluation
Fine-tunes `MahtaFetrat/Mana-Persian-Piper` (`SynthesizerTrn` + `MultiPeriodDiscriminator`) with 3 zero-initialized neural upgrades (`ProsodyEnhancedEmbedding` + `DurationPredictorAdapter` + `InvertibleFlowAttentionCoupling`) on `MahtaFetrat/Mana-TTS`, exports `output/persian_vits_upgraded.onnx`, and runs objective before/after evaluation.

In [ ]:
import os, subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'espeak-ng'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime', 'soundfile', 'scipy', 'transformers', 'pandas', 'pyarrow'], check=True)
if not os.path.isdir('/tmp/persian-vits-tts'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SoroushSoleimani/persian-vits-demo.git', '/tmp/persian-vits-tts'], check=True)
else:
    subprocess.run(['git', '-C', '/tmp/persian-vits-tts', 'pull', '--ff-only'], check=True)
%cd /tmp/persian-vits-tts
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!python piper_vits/monotonic_align/setup.py build_ext --inplace
!python test_pipeline.py

In [ ]:
!python finetune_persian.py

In [ ]:
!python evaluate_5_sentences.py
import shutil, glob
os.makedirs('/kaggle/working/output', exist_ok=True)
for pat in ['output/*.wav', 'output/*.png', 'output/*.json', 'output/*.onnx']:
    for src in glob.glob(pat):
        shutil.copy2(src, os.path.join('/kaggle/working/output', os.path.basename(src)))
print('Copied deliverables to /kaggle/working/output:', sorted(os.listdir('/kaggle/working/output')))